We use TDC datasets for several classification and regression tasks mentioned in the preprint, using either random or scaffold splits with three different random seeds to produce triplicates.

# 01. Generate Features

In [1]:
# pkgs
import h5py
import torch
import numpy as np

from pathlib import Path
from tqdm.notebook import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Crippen, Descriptors, Lipinski, MolSurf

# disable warnings
RDLogger.DisableLog('rdApp.*')

from intermol.main.inference import SAEInferenceConfig, SAEWithBaseModel

# defaults
MODEL_NAME = "ibm/MoLFormer-XL-both-10pct"
USE_MOLFORMER = True

MODEL_PATH = Path("./MoLFormer-XL")
DIRS = {
    int(pth.name.split("-")[1]): pth
    for pth in MODEL_PATH.glob("result-*-3072-128-*")
}

DATA_DIR = Path("data/tdc")
DATA_PTH = DATA_DIR / "cSMILES.txt"
OUTFN_PTH = DATA_DIR / "probe_features.h5"

## SAE config
HIDDEN_DIM = 3072
K = 128

In [2]:
# init data
## use extracted unique canonical SMILES across all TDC datasets
## to ensure correct SMILES ordering within 'OUTFN_PTH'
## when using TDC datasets split with different random seeds

with open(DATA_PTH, 'r') as h:
    data = [smi.rstrip("\n") for smi in h.readlines()]
n_data = len(data)

## 01A. FP & Physicochemical Descriptors

In [3]:
# init FP & Physicochemical Descriptors
rdk_fs = {
    "ECFP0": AllChem.GetMorganGenerator(radius=0, fpSize=2048),
    "ECFP2": AllChem.GetMorganGenerator(radius=1, fpSize=2048),
    "ECFP4": AllChem.GetMorganGenerator(radius=2, fpSize=2048),
    "PcDs": {
        "LogP": Crippen.MolLogP,
        "MR": Crippen.MolMR,
        "MW": Descriptors.MolWt,
        "FormalCharge": Chem.GetFormalCharge,
        "FractionCSP3": Lipinski.FractionCSP3,
        "HeavyAtomCount": Lipinski.HeavyAtomCount,
        "HBA": Lipinski.NumHAcceptors,
        "HBD": Lipinski.NumHDonors,
        "Rotatables": Lipinski.NumRotatableBonds,
        "RingCount": Lipinski.RingCount,
        "TPSA": MolSurf.TPSA,
        "AtomCount": lambda mol: mol.GetNumAtoms(onlyExplicit=False)
    }
}

# generate features
with h5py.File(OUTFN_PTH, 'a') as out_h5f:
    # RDKit-parsable checks
    is_parsable = np.fromiter(
        (Chem.MolFromSmiles(smi) is not None for smi in data),
        dtype=np.uint8,
        count=len(data)
    )
    out_h5f.create_dataset(
        "isSmilesRDKitParsable",
        data=is_parsable,
        compression="lzf"
    )

    # FP & Physicochemical
    for f_name, rdk_gen in tqdm(rdk_fs.items(), desc="Generating features..."):
        if f_name == "PcDs":
            arr = np.zeros((n_data, 12))
            for smi_i, smi in enumerate(data):
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    arr[smi_i, :] = [gen(mol) for gen in rdk_gen.values()]
        else:
            arr = np.zeros((n_data, 2048), dtype=np.uint8)
            for smi_i, smi in enumerate(data):
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    arr[smi_i, :] = rdk_gen.GetFingerprintAsNumPy(mol)

        # create dataset
        out_h5f.create_dataset(f_name, data=arr, compression="lzf")

    # clean up
    del is_parsable, arr

Generating features...:   0%|          | 0/4 [00:00<?, ?it/s]

## 01B. MolFormer & SAE Embeddings

In [4]:
# init SAEWithBaseModel
sae_cfgs = [
    SAEInferenceConfig(
        layer=layer,
        hidden_dim=HIDDEN_DIM,
        k=K,
        weights_path=dir_path / "norm" / "norm-sae_state-dict_250000s_28095744t.pt"
    ) for layer, dir_path in DIRS.items()
]
module = SAEWithBaseModel(sae_cfgs, MODEL_NAME, USE_MOLFORMER)

# inference
mf_hidden_dim = module.base_model.config.hidden_size
mf_shape = (len(DIRS), n_data, mf_hidden_dim)
mean_mf = torch.zeros(mf_shape, dtype=torch.float32)
max_mf = torch.zeros(mf_shape, dtype=torch.float32)

sae_shape = (len(DIRS), n_data, HIDDEN_DIM)
mean_sae = torch.zeros(sae_shape, dtype=torch.float32)
max_sae = torch.zeros(sae_shape, dtype=torch.float32)

with torch.no_grad():
    for smi_i, smi in tqdm(enumerate(data), total=n_data, desc="Generating features..."):
        out = module.encode_multi(smi, sae_cfgs)
        for l_i, cfg in enumerate(sae_cfgs):
            mf, sae = out[cfg.key]

            mf_l = mf.squeeze()[1:-1, :]
            mean_mf[l_i, smi_i, :] = mf_l.mean(dim=0)
            max_mf[l_i, smi_i, :] = mf_l.max(dim=0).values

            sae_l = sae.squeeze()[1:-1, :]
            mean_sae[l_i, smi_i, :] = sae_l.mean(dim=0)
            max_sae[l_i, smi_i, :] = sae_l.max(dim=0).values

# to numpy
mean_mf = mean_mf.cpu().numpy()
max_mf = max_mf.cpu().numpy()

mean_sae = mean_sae.cpu().numpy()
max_sae = max_sae.cpu().numpy()

# generate features
with h5py.File(OUTFN_PTH, 'a') as out_h5f:
    # create groups
    g_mf = out_h5f.require_group("MolFormer")
    g_mf_mean = g_mf.require_group("Mean")
    g_mf_max = g_mf.require_group("Max")

    g_sae = out_h5f.require_group("SAE")
    g_sae_mean = g_sae.require_group("Mean")
    g_sae_max = g_sae.require_group("Max")

    # create datasets
    for l_i, layer in enumerate(DIRS):
        g_mf_mean.create_dataset(
            str(layer), data=mean_mf[l_i, :, :], compression="lzf"
        )
        g_mf_max.create_dataset(
            str(layer), data=max_mf[l_i, :, :], compression="lzf"
        )

        g_sae_mean.create_dataset(
            str(layer), data=mean_sae[l_i, :, :], compression="lzf"
        )
        g_sae_max.create_dataset(
            str(layer), data=max_sae[l_i, :, :], compression="lzf"
        )

Generating features...:   0%|          | 0/50813 [00:00<?, ?it/s]

# 02. Linear Probing

In [1]:
# pkgs
import h5py
import numpy as np
import polars as pl

from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.linear_model import LogisticRegressionCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    make_scorer,
    accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    root_mean_squared_error
)
from scipy.stats import spearmanr

import warnings
# ignore warnings
warnings.filterwarnings('ignore')

# defaults
DATA_DIR = Path("data/tdc")
DATA_PTH = DATA_DIR / "cSMILES.txt"
FEATURES_PTH = DATA_DIR / "probe_features.h5"
OUTFN_PTH = DATA_DIR / "linear_probes_result.parquet"

FEATURE_GROUP = {
    "RDKit": ["ECFP0", "ECFP2", "ECFP4", "PcDs"],
    "MolFormer": [
        "MolFormer/Mean/1", "MolFormer/Mean/3", "MolFormer/Mean/6", "MolFormer/Mean/9", "MolFormer/Mean/12",
        "MolFormer/Max/1", "MolFormer/Max/3", "MolFormer/Max/6", "MolFormer/Max/9", "MolFormer/Max/12"
    ],
    "SAE": [
        "SAE/Mean/1", "SAE/Mean/3", "SAE/Mean/6", "SAE/Mean/9", "SAE/Mean/12",
        "SAE/Max/1", "SAE/Max/3", "SAE/Max/6", "SAE/Max/9", "SAE/Max/12"
    ]
}

# init data
with open(DATA_PTH, 'r') as h:
    data_map = {smi.rstrip("\n"): smi_i for smi_i, smi in enumerate(h.readlines())}
n_data = len(data_map)

In [2]:
# probing latents
out = []
for d_i, data_pth in enumerate(DATA_DIR.glob("tdc_dataset_*.parquet")):
    # parse dataset
    dataset = pl.read_parquet(data_pth)
    # map cSMILES to their indices
    dataset = dataset.with_columns(
        pl.col("canon_smiles").replace_strict(data_map).alias("canon_index")
    )

    # get available tasks
    tasks = dataset["task"].unique(maintain_order=True).to_list()

    # gather features
    with h5py.File(FEATURES_PTH, 'r') as out_h5f:
        # RDK features sanity check
        chk = out_h5f["isSmilesRDKitParsable"][:]
        chkptr = np.flatnonzero(chk == 0).tolist()

        if len(chkptr) > 0:
            warnings.warn(
                f"Found {len(chkptr)} RDKit-unparsable SMILES. "
                "These samples will be excluded from RDKit-based feature evaluation."
            )

        for fg, fs_name in tqdm(FEATURE_GROUP.items(), desc="Processing per group..."):
            for task in tqdm(tasks, desc="Processing per task...", leave=False):
                if fg == "RDKit":
                    subset = dataset.filter(
                        (pl.col("task") == task)
                        & (~pl.col("canon_index").is_in(chkptr))
                    ).sort("canon_index")
                else:
                    subset = dataset.filter(pl.col("task") == task).sort("canon_index")

                task_type = subset["task_type"].unique().item()
                label_dtype = np.int8 if task_type == "cls" else np.float64

                # gather train-test splits
                cond = (pl.col("split") == "test")
                subset_train = subset.filter(~cond)
                subset_test = subset.filter(cond)

                label_train = subset_train["y"].to_numpy().astype(label_dtype)
                label_test = subset_test["y"].to_numpy().astype(label_dtype)

                ptr_train = subset_train["canon_index"].to_numpy()
                ptr_test = subset_test["canon_index"].to_numpy()

                for f_name in fs_name:
                    split_train = out_h5f[f_name][ptr_train]
                    split_test = out_h5f[f_name][ptr_test]

                    # normalize X values for PcDs
                    if f_name == "PcDs":
                        scaler = StandardScaler()
                        split_train = scaler.fit_transform(split_train)
                        split_test = scaler.transform(split_test)
                        del scaler

                    if f_name.startswith("MolFormer") or f_name.startswith("SAE"):
                        model_name, pooling, layer = f_name.split("/")
                        method = model_name + f" ({pooling})"
                        layer = int(layer)
                    else:
                        method = f_name
                        layer = None

                    # model
                    if task_type == "cls":
                        model = LogisticRegressionCV(
                            Cs=5,
                            scoring=make_scorer(
                                roc_auc_score, response_method="predict_proba"
                            ),
                            cv=5,
                            n_jobs=-1,
                            max_iter=1000
                        )
                        model.fit(split_train, label_train)

                        # test
                        y_pred = model.predict(split_test)
                        y_pred_proba = model.predict_proba(split_test)[:, 1]

                        out.append({
                            "Task": task,
                            "Method": method,
                            "Layer": layer,
                            "Rep": d_i + 1,
                            "HPs": model.C_[0],
                            "Coef": model.coef_.flatten().tolist(),
                            "Intercept": model.intercept_[0],
                            "Accuracy": accuracy_score(label_test, y_pred),
                            "MCC": matthews_corrcoef(label_test, y_pred),
                            "ROC-AUC": roc_auc_score(label_test, y_pred_proba)
                        })

                    elif task_type == "reg":
                        model = RidgeCV(
                            alphas=np.logspace(-4, 4, 5),
                            scoring="neg_root_mean_squared_error",
                            cv=5
                        )
                        model.fit(split_train, label_train)

                        # test
                        y_pred = model.predict(split_test)

                        out.append({
                            "Task": task,
                            "Method": method,
                            "Layer": layer,
                            "Rep": d_i + 1,
                            "HPs": model.alpha_,
                            "Coef": model.coef_.flatten().tolist(),
                            "Intercept": model.intercept_,
                            "RMSE": root_mean_squared_error(label_test, y_pred),
                            "Spearman": spearmanr(label_test, y_pred).statistic
                        })

                    else:
                        raise ValueError("Unknown 'task_type'. " \
                        "Available 'task_type' are 'cls' and 'reg'.")

schema = {
    "Task": pl.String,
    "Method": pl.String,
    "Layer": pl.UInt8,
    "Rep": pl.UInt8,
    "HPs": pl.Float32,
    "Coef": pl.List(pl.Float32),
    "Intercept": pl.Float32,
    "Accuracy": pl.Float32,
    "MCC": pl.Float32,
    "ROC-AUC": pl.Float32,
    "RMSE": pl.Float32,
    "Spearman": pl.Float32
}
out_df = pl.DataFrame(out, schema)
out_df.write_parquet(OUTFN_PTH)

Processing per group...:   0%|          | 0/3 [00:00<?, ?it/s]

Processing per task...:   0%|          | 0/32 [00:00<?, ?it/s]

Processing per task...:   0%|          | 0/32 [00:00<?, ?it/s]

Processing per task...:   0%|          | 0/32 [00:00<?, ?it/s]

Processing per group...:   0%|          | 0/3 [00:00<?, ?it/s]

...

Processing per task...:   0%|          | 0/32 [00:00<?, ?it/s]

Processing per group...:   0%|          | 0/3 [00:00<?, ?it/s]

Processing per task...:   0%|          | 0/32 [00:00<?, ?it/s]

Processing per task...:   0%|          | 0/32 [00:00<?, ?it/s]

Processing per task...:   0%|          | 0/32 [00:00<?, ?it/s]

# 03. For Interp.

In [ ]:
import numpy as np
import polars as pl

df = pl.read_parquet("data/tdc/linear_probes_result.parquet")

# standardize probe coef
zs_df = df.with_columns(
    pl.col("Coef").list.eval(
        (pl.element() - pl.element().mean()) / pl.element().std()
    )
    .alias("ZScore")
)

# agg
zs_agg_df = (
    zs_df
    .group_by(["Task", "Method", "Layer"], maintain_order=True)
    .agg("ZScore")
)

# avg across triplicates
metric = {
    "MeanZ": [],
    "StdZ": [],
    "SortedIndex": [],
    "SortedMeanZ": [],
    "SortedStdZ": [],
}
for r in zs_agg_df.iter_rows(named=True):
    zs = np.array(r["ZScore"])
    mzs = zs.mean(axis=0)
    stdzs = zs.std(axis=0)
    sort_idxs = mzs.argsort()[::-1]
    sort_mzs = mzs[sort_idxs]
    sort_stdzs = stdzs[sort_idxs]

    metric['MeanZ'].append(mzs)
    metric['StdZ'].append(stdzs)
    metric['SortedIndex'].append(sort_idxs)
    metric['SortedMeanZ'].append(sort_mzs)
    metric['SortedStdZ'].append(sort_stdzs)
metric_df = pl.DataFrame(metric)
zs_m_df = pl.concat([zs_agg_df, metric_df], how="horizontal")

zs_m_df.write_parquet("data/tdc/linear_probes_interp.parquet")

C:\Users\kenne\AppData\Local\Temp\ipykernel_15888\3649686692.py:43: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  zs_m_df = pl.concat([zs_agg_df, metric_df], how="horizontal")
